# LLM Fine-Tuning — Practice Lab

**Goal:** Fine-tune **Qwen2.5-0.5B-Instruct** with **Unsloth + QLoRA** on Alpaca data.

**You will:**
1. Load a tiny 4-bit model and attach **LoRA** adapters
2. Format instruction data into chat text
3. Train with **SFTTrainer**
4. Compare answers **before vs after** fine-tuning
5. Measure improvement with **BLEU** and **cosine similarity**

Run cells **top to bottom**. This model is small (~0.5B) so training is fast on a laptop GPU.


## Quick concepts

| Idea | In this lab |
|------|-------------|
| **Small LLM** | Qwen2.5-0.5B — fast to train, low VRAM |
| **QLoRA** | 4-bit base weights + LoRA adapters |
| **SFT** | Supervised fine-tuning on instruction → answer pairs |
| **BLEU** | N-gram overlap with reference answers (higher = closer wording) |
| **Cosine sim.** | Semantic similarity of sentence embeddings (higher = closer meaning) |


### Setup

In [1]:
# Run once in a new environment, then restart the kernel if needed:
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" xformers trl accelerate bitsandbytes peft datasets transformers sacrebleu sentence-transformers


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 111.5 MB/s eta 0:

**Note:** you may need to restart the kernel to use updated packages.

In [2]:
import os
import time

os.environ.setdefault("WANDB_DISABLED", "true")

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TextStreamer
from trl import SFTConfig, SFTTrainer

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("VRAM (GB):", round(props.total_memory / 1024**3, 2))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM (GB): 14.56


## 1. Load model + LoRA

`unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit` fits easily on consumer GPUs.


In [3]:
MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
LORA_R = 32

t0 = time.time()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
print(f"Model loaded in {time.time() - t0:.1f}s")

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_R,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable/1e6:.2f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)")


==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded in 22.9s


Unsloth 2026.6.9 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Trainable: 17.60M / 332.7M (5.29%)


## 2. Prepare dataset

`yahma/alpaca-cleaned` → chat-formatted `text` column.


In [4]:
def alpaca_user_text(example):
    user_text = example["instruction"].strip()
    if example.get("input") and example["input"].strip():
        user_text += "\n\nInput:\n" + example["input"].strip()
    return user_text


def alpaca_to_chat(example):
    messages = [
        {"role": "user", "content": alpaca_user_text(example)},
        {"role": "assistant", "content": example["output"].strip()},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    }


EVAL_SIZE = 30
raw_ds = load_dataset("yahma/alpaca-cleaned", split="train")
shuffled = raw_ds.shuffle(seed=3407)
train_dataset = shuffled.select(range(2000)).map(alpaca_to_chat)
eval_raw = shuffled.select(range(2000, 2000 + EVAL_SIZE))
eval_examples = [
    {"prompt": alpaca_user_text(ex), "reference": ex["output"].strip()}
    for ex in eval_raw
]

print("Train size:", len(train_dataset))
print("Eval size:", len(eval_examples))
print(train_dataset[0]["text"][:500])


README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Train size: 2000
Eval size: 30
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
List 5 popular dishes in US.<|im_end|>
<|im_start|>assistant
1. Hamburger: A classic American dish consisting of a beef patty served on a bun, often topped with cheese and various toppings such as lettuce, tomatoes, onions, and condiments like ketchup and mustard.

2. Macaroni & Cheese: A comforting dish made from elbow macaroni mixed with a creamy cheese sauce and often baked until


## 3. Baseline (before training)

Run the same prompts before and after fine-tuning to compare quality.


In [5]:
FastLanguageModel.for_inference(model)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def generate_answer(prompt_text, max_new_tokens=128):
    messages = [{"role": "user", "content": prompt_text}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(DEVICE)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )
    new_tokens = output_ids[0, inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("Generating baseline answers on held-out eval set...")
baseline_outputs = [generate_answer(ex["prompt"]) for ex in eval_examples]

test_prompts = [
    "List 3 healthy breakfast ideas with nutritional benefits.",
    "Write a Python function that reverses a string.",
]

streamer = TextStreamer(tokenizer, skip_prompt=True)

for prompt_text in test_prompts:
    print("\n" + "=" * 60)
    print(f"PROMPT: {prompt_text}")
    print("=" * 60)
    messages = [{"role": "user", "content": prompt_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(DEVICE)
    _ = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        streamer=streamer,
    )


Generating baseline answers on held-out eval set...


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12


PROMPT: List 3 healthy breakfast ideas with nutritional benefits.
Certainly! Here are three healthy breakfast ideas along with their nutritional benefits:

1. **Smoothie Bowls:**
   - Smoothies are a great way to start your day without any added sugars or unhealthy fats.
   - They can be topped with fruits like bananas, berries, and spinach for vitamins and minerals.

2. **Smoothie Packs:**
   - Prepare a pack of smoothies in the refrigerator to ensure they remain fresh throughout the day.
   - Each serving typically contains around 50-70 calories and 20-30 grams of protein.

3. **Oatmeal Pancakes:**
   - Oats are rich in fiber, protein, and other nutrients that keep you feeling full.
   - Use uns


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROMPT: Write a Python function that reverses a string.
Certainly! Below is a Python function that reverses a given string:

```python
def reverse_string(s):
    """
    Reverses the characters of the input string s.
    
    Args:
        s (str): The string to be reversed.
        
    Returns:
        str: The reversed string.
    """
    return s[::-1]

# Example usage:
original_string = "Hello, World!"
reversed_string = reverse_string(original_string)
print(reversed_string)  # Output: "!dlroW ,olleH"
```

### Explanation:
- **Input:** A `string` or any data type that can be converted into a string.
  
- **Function:** A function named `reverse_string`.

- **Return Type:**


## 4. Train


In [6]:
FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        output_dir="qwen05b_qlora_outputs",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=150,
        learning_rate=3e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        report_to="none",
    ),
)

t0 = time.time()
trainer.train()
print(f"Training time: {(time.time() - t0) / 60:.1f} min")


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 17,596,416 of 511,629,184 (3.44% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.853403
20,1.339244
30,1.329706
40,1.236287
50,1.262558
60,1.260660
70,1.231789
80,1.249706
90,1.286283
100,1.280856


Unsloth: Restored added_tokens_decoder metadata in qwen05b_qlora_outputs/checkpoint-150/tokenizer_config.json.


Training time: 4.1 min


## 5. Inference after fine-tuning

Run the **same prompts** as in step 3 and compare the answers.


In [7]:
FastLanguageModel.for_inference(model)

for prompt_text in test_prompts:
    print("\n" + "=" * 60)
    print(f"PROMPT: {prompt_text}")
    print("=" * 60)
    messages = [{"role": "user", "content": prompt_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(DEVICE)
    _ = model.generate(
        **inputs,
        # max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        streamer=streamer,
    )


PROMPT: List 3 healthy breakfast ideas with nutritional benefits.


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


1. Avocado Toast: A nutritious and filling breakfast option that includes avocado, which is rich in healthy fats, vitamins, and minerals, including vitamin E, magnesium, and B vitamins.

2. Smoothie Bowl: A delicious and healthy breakfast idea using fruits, vegetables, protein sources, and dairy or non-dairy milk to create an energizing and balanced meal.

3. Greek Yogurt Parfait: A tasty and healthy breakfast idea combining Greek yogurt, fresh fruit, and granola for a satisfying and nutrient-packed breakfast.<|im_end|>

PROMPT: Write a Python function that reverses a string.
Here is one possible implementation of the `reverse_string` function in Python:

```python
def reverse_string(string):
    """
    This function takes a string as input and returns its reversed version.
    
    Args:
        string (str): The string to be reversed
    
    Returns:
        str: The reversed string
    """
    return string[::-1]
```

This function works by using Python's slicing feature (`[::-1]`

## 6. Quantitative evaluation

Score model answers against **reference** outputs from the held-out eval set:

- **BLEU** — token overlap with the gold answer (classic MT metric)
- **Cosine similarity** — embedding similarity via `all-MiniLM-L6-v2` (captures paraphrases BLEU may miss)

Higher scores after fine-tuning suggest the model is closer to the expected answers.

In [8]:
import numpy as np
import pandas as pd
from sacrebleu.metrics import BLEU
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

bleu = BLEU()
references = [ex["reference"] for ex in eval_examples]

print("Generating fine-tuned answers on eval set...")
finetuned_outputs = [generate_answer(ex["prompt"]) for ex in eval_examples]

bleu_before = bleu.corpus_score(baseline_outputs, [references]).score
bleu_after = bleu.corpus_score(finetuned_outputs, [references]).score

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def mean_cosine(preds, refs):
    pred_emb = embedder.encode(preds, convert_to_numpy=True, show_progress_bar=False)
    ref_emb = embedder.encode(refs, convert_to_numpy=True, show_progress_bar=False)
    sims = [
        cosine_similarity(pred_emb[i : i + 1], ref_emb[i : i + 1])[0, 0]
        for i in range(len(preds))
    ]
    return float(np.mean(sims))


cos_before = mean_cosine(baseline_outputs, references)
cos_after = mean_cosine(finetuned_outputs, references)

summary = pd.DataFrame(
    {
        "metric": ["BLEU", "cosine similarity"],
        "before": [bleu_before, cos_before],
        "after": [bleu_after, cos_after],
        "delta": [bleu_after - bleu_before, cos_after - cos_before],
    }
)
display(summary.style.format({"before": "{:.3f}", "after": "{:.3f}", "delta": "{:+.3f}"}))

example_idx = 0
pd.DataFrame(
    [
        {
            "prompt": eval_examples[example_idx]["prompt"][:100] + "...",
            "reference": eval_examples[example_idx]["reference"][:180] + "...",
            "before": baseline_outputs[example_idx][:180] + "...",
            "after": finetuned_outputs[example_idx][:180] + "...",
        }
    ]
)

Generating fine-tuned answers on eval set...


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,metric,before,after,delta
0,BLEU,6.663,6.919,+0.256
1,cosine similarity,0.653,0.698,+0.045


,prompt,reference,before,after
0,Classify the following sentence into either an...,"The sentence ""Please turn off the lights"" is a...","The input ""Please turn off the lights."" is cla...",The sentence is an **interrogative** sentence....


In [9]:
save_path = "qwen05b_lora_adapters"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("Saved adapters to:", save_path)

if torch.cuda.is_available():
    peak_gb = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Peak VRAM: {peak_gb:.2f} GB")


Unsloth: Restored added_tokens_decoder metadata in qwen05b_lora_adapters/tokenizer_config.json.


Saved adapters to: qwen05b_lora_adapters
Peak VRAM: 1.25 GB


## Practice exercises

1. Swap model to `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit` — compare quality and speed vs the default **0.5B** model (same Qwen family + chat template; reliable on Colab T4).
2. Try `LORA_R = 8` vs `32` — which trains faster?
3. Add your own prompts to `test_prompts` and compare before/after outputs.
4. Train on 500 vs 2000 samples — note the effect on BLEU / cosine scores.
5. Increase `EVAL_SIZE` to 100 — are the metrics more stable?

**Deliverable:** short write-up with training time, peak VRAM, BLEU/cosine before vs after, and one qualitative example.


## Exercise Implementation
We will now re-initialize the model and trainer with the new parameters requested.

In [10]:
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
LORA_R = 8 # Try 8 first, then change to 32 to compare speed

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_R,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [11]:
EVAL_SIZE = 100
TRAIN_SAMPLES = 500 # Toggle between 500 and 2000 for your comparison

shuffled = raw_ds.shuffle(seed=3407)
train_dataset = shuffled.select(range(TRAIN_SAMPLES)).map(alpaca_to_chat)
eval_raw = shuffled.select(range(TRAIN_SAMPLES, TRAIN_SAMPLES + EVAL_SIZE))
eval_examples = [
    {"prompt": alpaca_user_text(ex), "reference": ex["output"].strip()}
    for ex in eval_raw
]

test_prompts = [
    "Explain the concept of photosynthesis to a 5-year-old.",
    "Write a short sci-fi story about a robot that discovers feelings.",
    "What are the top 3 ways to stay productive while working from home?"
]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [12]:
FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        output_dir="qwen1.5b_outputs",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=100, # Kept short for quick exercise testing
        learning_rate=3e-4,
        logging_steps=10,
        optim="adamw_8bit",
        seed=3407,
        report_to="none",
    ),
)

t0 = time.time()
trainer.train()
training_time = (time.time() - t0) / 60
print(f"Training time for 1.5B model: {training_time:.1f} min")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 2 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 9,232,384 of 1,552,946,688 (0.59% trained)


Step,Training Loss
10,1.841070
20,1.166585
30,1.078634
40,1.102860
50,1.015688
60,1.132383
70,1.061515
80,1.008449
90,0.987273
100,1.082670


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in qwen1.5b_outputs/checkpoint-100/tokenizer_config.json.


Training time for 1.5B model: 3.4 min
